# M5 — Trading strategy and simulation

Use the fixed model choice and strategy rules. Long-only monthly portfolios hold up to three eligible stocks above the configured probability threshold, equally weighted, otherwise cash. Cash return is zero. Compare with hand-written momentum and buy-and-hold benchmarks.

Run cells in order with the project environment (`uv run jupyter lab`). Paths support the project root or `notebooks/`. See [workshop order](../_docs/workshop_order.md) for dependencies and checkpoints. Numerical outputs come from actual local inputs; no synthetic data are substituted for market results.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "config/project.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/project.yaml").exists(), "Launch from the project root or notebooks/"
from uranium_signals.config import load_config
from uranium_signals.universe import load_universe
from uranium_signals.providers import acquire
from uranium_signals.run import research_tables
cfg = load_config(ROOT / "config/project.yaml")
universe = load_universe(cfg)
tickers = set(universe.ticker) | {cfg["benchmark"], cfg["broad_benchmark"]} | set(cfg["optional_benchmarks"])


In [ ]:
prices, macro, source_metadata = acquire(cfg, tickers, offline=True)
prices, quality, targets, table = research_tables(cfg, prices, macro, universe)


## 1. Form and execute a signal

At month-end, rank probabilities and set target weights. Enter at the first benchmark trading session's adjusted open in the following month. Exit/rebalance at the next month's first adjusted open. Missing execution prices for a held security are material errors; incomplete final periods are excluded.

In [ ]:
from uranium_signals.run import evaluate
results = evaluate(cfg, table, targets)
predictions, audits, metrics, calibration, selected, portfolio_returns, economics, model_trades, momentum_trades, baseline_members = results
first_date = model_trades.date.min()
ranked = predictions[(predictions.model == selected) & predictions.date.eq(first_date)].sort_values(["probability", "ticker"], ascending=[False, True])
display(ranked[["date", "ticker", "probability", "execution_date", "exit_date", "trade_return"]])
display(model_trades.head())


## 2. Explain turnover and costs

Let target weights be w and the previous portfolio's drifted weights be v. Traded value is `sum(abs(w-v))`; a full switch from one stock to another trades two dollars per dollar of portfolio value. Cost equals traded value times the per-side basis-point assumption. Net return is `(1-cost)*(1+gross_return)-1`. Initial purchases are charged; final liquidation is not imposed. One-way turnover is half the absolute traded weight.

In [ ]:
cost_rate = cfg["transaction_cost_bps"] / 10000
assert np.allclose(model_trades.cost_fraction, model_trades.traded_value * cost_rate)
assert np.allclose(model_trades["return"], (1 - model_trades.cost_fraction) * (1 + model_trades.gross_return) - 1)
assert (model_trades.execution_date > model_trades.date).all()
print("Costs in bps per dollar bought or sold:", cfg["transaction_cost_bps"])
print("Buy-and-hold inception members:", baseline_members)


## 3. Compare economic outcomes

Buy-and-hold buys once at holdout inception and lets weights drift; unavailable newer listings are excluded at inception. Benchmark funds use the same executable holding intervals. The annualization assumes 12 monthly periods/year, and Sharpe uses the configured fixed risk-free assumption. Performance reflects the documented fixed current-constituent study.

In [ ]:
display(economics)
wealth = (1 + portfolio_returns.set_index("date")).cumprod()
wealth.plot(title="Holdout wealth after assumed costs", figsize=(10, 5))
plt.show()
drawdown = wealth / wealth.cummax().clip(lower=1) - 1
drawdown.plot(title="Monthly holdout drawdown", figsize=(10, 4))
plt.show()
display(model_trades.nsmallest(5, "return"))


## Written-result checkpoint

Record absolute and benchmark-relative performance, drawdown, turnover, costs, trades, and the worst periods. Inspect one portfolio switch by hand. Explain why current-constituent bias, uncertain macro vintages, and the close/open interval difference limit the conclusions. Do not revise rules after seeing the holdout.